# 📘 Tutorial: Hiểu và Visualize Toàn diện Output của COLMAP (SfM)

Notebook này được thiết kế để giúp bạn **học và hiểu cặn kẽ bản chất cấu trúc dữ liệu mà COLMAP (Structure-from-Motion) xuất ra**, kèm theo các đoạn code visualization trực quan cho từng thành phần (chưa cần đụng tới bất kỳ bước xử lý hay lọc hậu kỳ nào).

---

### 📑 Mục lục bài học:
1. **Tổng quan về 3 file text cốt lõi của COLMAP** (`cameras.txt`, `images.txt`, `points3D.txt`)
2. **Khám phá `cameras.txt`**: Ma trận nội hàm (Intrinsics), Tiêu cự $f$, Tâm ảnh $(c_x, c_y)$, và Độ méo thấu kính $k_1$
3. **Khám phá `images.txt`**: Ngoại hàm (Extrinsics), Góc xoay Quaternion $\to$ Tọa độ Camera thật $C = -R^T T$
4. **Khám phá Feature Tracks (Dấu vết 2D-3D)**: Xem các điểm $(u, v)$ trên ảnh thật và phân biệt điểm đã nối 3D vs điểm chưa nối
5. **Khám phá `points3D.txt`**: Tọa độ $(x, y, z)$, Màu RGB gốc, và Sai số chiếu lại (Reprojection Error)
6. **Visualize 3D Toàn cảnh**: Vẽ đám mây điểm RGB + Hình nón camera (Camera Frustums) + Các tia nhìn hội tụ (Triangulation Rays)

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation

# Cấu hình font và hiển thị đồ thị
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print("✅ Môi trường Python đã sẵn sàng!")

## 1. Bản chất: COLMAP xuất ra cái gì?

Sau khi chạy thuật toán SfM (Structure-from-Motion) trên tập ảnh UAV, COLMAP trả về một mô hình hình học gồm **3 bảng dữ liệu chính**:

| File | Tên khái niệm | Ý nghĩa vật lý |
| :--- | :--- | :--- |
| `cameras.txt` | **Camera Intrinsics** | Đặc tính quang học của ống kính camera (tiêu cự, tâm ảnh, độ cong thấu kính). |
| `images.txt` | **Camera Extrinsics & 2D Observations** | Vị trí và góc nghiêng của Drone lúc chụp từng bức ảnh + Bảng tọa độ pixel $(u, v)$ của các điểm đặc trưng. |
| `points3D.txt` | **3D Sparse Points & Tracks** | Tọa độ không gian 3D $(x, y, z)$, màu sắc RGB, và danh sách các ảnh nhìn thấy điểm đó. |

Hãy cùng mổ xẻ từng file dưới đây:

## 2. Khám phá `cameras.txt` (Camera Intrinsics)

Mỗi dòng trong `cameras.txt` đại diện cho 1 cấu hình camera:
```text
# CAMERA_ID  MODEL          WIDTH  HEIGHT  PARAMS[f, cx, cy, k1]
  1          SIMPLE_RADIAL  1320   989     925.7016 660.0 494.5 0.0124
```

- **$f = 925.7$ px**: Tiêu cự camera tính theo pixel.
- **$c_x = 660.0, c_y = 494.5$ px**: Tâm quang học (thường là chính giữa ảnh $1320/2, 989/2$).
- **$k_1$**: Hệ số méo xuyên tâm (Radial Distortion). $k_1 > 0$ là méo phồng (barrel), $k_1 < 0$ là méo lõm (pincushion).

In [ ]:
# Mô phỏng đọc thông số từ cameras.txt
camera_model = {
    "id": 1,
    "model": "SIMPLE_RADIAL",
    "width": 1320,
    "height": 989,
    "f": 925.7016,
    "cx": 660.0,
    "cy": 494.5,
    "k1": 0.015  # Hệ số méo thấu kính mẫu
}

# Ma trận nội hàm K (Camera Intrinsics Matrix 3x3)
K = np.array([
    [camera_model["f"], 0, camera_model["cx"]],
    [0, camera_model["f"], camera_model["cy"]],
    [0, 0, 1]
])

print("📐 Ma trận Camera Intrinsics K (3x3):")
print(K)

# Visualize hiệu ứng méo thấu kính k1 tác động lên lưới pixel
gx, gy = np.meshgrid(np.linspace(0, camera_model["width"], 25), np.linspace(0, camera_model["height"], 20))
u_norm = (gx - camera_model["cx"]) / camera_model["f"]
v_norm = (gy - camera_model["cy"]) / camera_model["f"]
r2 = u_norm**2 + v_norm**2
u_dist = gx + (gx - camera_model["cx"]) * camera_model["k1"] * r2
v_dist = gy + (gy - camera_model["cy"]) * camera_model["k1"] * r2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(gx, gy, c='blue', s=8)
ax1.set_title("Lưới Pixel Lý Tưởng (Không méo thấu kính)")
ax1.set_xlim(0, camera_model["width"]); ax1.set_ylim(camera_model["height"], 0)
ax1.set_aspect('equal'); ax1.grid(True, linestyle=':', alpha=0.5)

ax2.scatter(u_dist, v_dist, c='red', s=8)
ax2.set_title(f"Lưới Pixel Sau Méo Radial (k1 = {camera_model['k1']})")
ax2.set_xlim(0, camera_model["width"]); ax2.set_ylim(camera_model["height"], 0)
ax2.set_aspect('equal'); ax2.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Khám phá `images.txt` (Camera Extrinsics & 2D Observations)

Trong `images.txt`, cứ **2 dòng liên tiếp** sẽ mô tả đầy đủ 1 bức ảnh chụp:

- **Dòng 1 (Extrinsics Pose)**: `IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME`
  - $[q_w, q_x, q_y, q_z]$: Quaternion biểu diễn góc quay camera $R$.
  - $[t_x, t_y, t_z]$: Vector tịnh tiến $T$ từ thế giới sang camera.
  - ⚠️ **Lưu ý quan trọng**: Tọa độ thật của Drone trong không gian không phải là $T$, mà là: **$C = -R^T \cdot T$**.

- **Dòng 2 (2D Points & Track Links)**: Danh sách bộ ba `X_pixel, Y_pixel, POINT3D_ID`
  - `POINT3D_ID = -1`: Điểm đặc trưng 2D tìm thấy nhưng không khớp được với ảnh khác (không tạo thành điểm 3D).
  - `POINT3D_ID >= 0`: Điểm 2D đã được **triangulate thành công** thành điểm 3D có ID tương ứng!

In [ ]:
# Minh họa tính toán tọa độ thật của Drone từ Quaternion và Translation
sample_image_record = {
    "image_id": 1,
    "name": "001.png",
    "qvec": np.array([0.8535, -0.1464, 0.3535, 0.3535]), # [qw, qx, qy, qz]
    "tvec": np.array([-12.4, 4.8, 45.2])                 # [tx, ty, tz]
}

# 1. Đổi Quaternion -> Ma trận quay Rotation Matrix R (3x3)
r_quat = sample_image_record["qvec"]
rot_matrix = Rotation.from_quat([r_quat[1], r_quat[2], r_quat[3], r_quat[0]]).as_matrix()

# 2. Tính tọa độ Drone thực tế trong thế giới thực (World Coordinates)
camera_center_world = -rot_matrix.T @ sample_image_record["tvec"]

print(f"📷 Ảnh: {sample_image_record['name']} (ID: {sample_image_record['image_id']})")
print(f"   • Ma trận quay R (3x3):\n{rot_matrix}")
print(f"   • Vector Tịnh tiến T (World -> Cam): {sample_image_record['tvec']}")
print(f"   📍 Tọa độ GPS/World thật của Drone C = -R^T * T: {camera_center_world.round(2)} mét")

## 4. Khám phá Điểm 2D trên ảnh: Matched vs Unmatched Points

Hãy vẽ thử toàn bộ các điểm đặc trưng 2D mà COLMAP trích xuất được trên một bức ảnh để thấy sự khác biệt giữa điểm có liên kết 3D (`POINT3D_ID >= 0`) và điểm bị loại (`POINT3D_ID = -1`).

In [ ]:
np.random.seed(101)
n_features = 300

# Giả lập tọa độ 2D của các điểm đặc trưng trích xuất trên ảnh
u_pts = np.random.uniform(50, camera_model["width"] - 50, n_features)
v_pts = np.random.uniform(50, camera_model["height"] - 50, n_features)

# 70% số điểm được khớp 3D thành công (point3D_id >= 0), 30% bị loại (point3D_id = -1)
point3d_ids = np.array([i if np.random.rand() > 0.3 else -1 for i in range(n_features)])

matched_mask = point3d_ids >= 0
unmatched_mask = point3d_ids == -1

plt.figure(figsize=(13, 7))
# Giả lập khung hình ảnh UAV (nền tối)
plt.imshow(np.ones((camera_model["height"], camera_model["width"], 3), dtype=np.uint8) * 40)

# Vẽ các điểm không tạo được 3D (màu xám nhạt)
plt.scatter(u_pts[unmatched_mask], v_pts[unmatched_mask], c='gray', s=20, alpha=0.5, 
            label=f"Unmatched 2D Keypoints (ID = -1): {np.sum(unmatched_mask)} điểm")

# Vẽ các điểm triangulate thành công ra 3D (màu xanh lá nổi bật)
plt.scatter(u_pts[matched_mask], v_pts[matched_mask], c='lime', s=45, edgecolors='black', linewidth=0.5,
            label=f"Triangulated 3D Points (ID >= 0): {np.sum(matched_mask)} điểm")

plt.title(f"Trực quan hóa Dòng 2 của images.txt: Phân bố 2D Keypoints trên Frame {sample_image_record['name']}", fontsize=12)
plt.xlabel("Pixel U"); plt.ylabel("Pixel V")
plt.xlim(0, camera_model["width"]); plt.ylim(camera_model["height"], 0) # Gốc (0,0) ở góc trên bên trái
plt.legend(loc='upper right', framealpha=0.9)
plt.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Khám phá `points3D.txt` (Điểm 3D, Sai số và Độ dài Track)

Mỗi dòng trong `points3D.txt` đại diện cho 1 điểm 3D đã được tính toán trong không gian:
```text
# POINT3D_ID  X       Y       Z       R    G    B    ERROR  TRACK[IMAGE_ID, POINT2D_IDX, ...]
  1402        -15.42  3.18    120.45  180  175  160  0.48   1 42 5 18 12 99
```

### 2 Chỉ số chất lượng quan trọng nhất của COLMAP:
1. **`ERROR` (Reprojection Error)**: Sai số hình chiếu tính bằng pixel (khoảng cách giữa điểm 2D thực tế và hình chiếu của điểm 3D). Giá trị $< 1.0$ px là rất tốt.
2. **`TRACK LENGTH` (Độ dài dấu vết)**: Điểm 3D này được nhìn thấy bởi bao nhiêu camera (ví dụ: `1 42 5 18 12 99` $\rightarrow$ được 3 camera số 1, 5, 12 cùng quan sát). Track càng dài thì tọa độ 3D càng chính xác.

In [ ]:
# Phân tích phân phối thống kê chất lượng của 3D Points từ COLMAP
np.random.seed(42)
n_pts_demo = 5000

# Giả lập phân phối sai số Reprojection Error (thường lệch phải, tập trung quanh 0.3 - 0.8 px)
reproj_errors = np.random.gamma(shape=2.5, scale=0.25, size=n_pts_demo)

# Giả lập phân phối Track Length (số camera cùng nhìn 1 điểm, từ 2 đến 25 camera)
track_lengths = np.random.geometric(p=0.2, size=n_pts_demo) + 1
track_lengths = np.clip(track_lengths, 2, 35)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram 1: Reprojection Error
ax1.hist(reproj_errors, bins=40, color='#e74c3c', edgecolor='black', alpha=0.75)
ax1.axvline(np.mean(reproj_errors), color='blue', linestyle='--', linewidth=2, label=f"Mean Error: {np.mean(reproj_errors):.2f} px")
ax1.axvline(1.0, color='black', linestyle=':', linewidth=2, label="Ngưỡng chuẩn (1.0 px)")
ax1.set_title("Phân phối Sai số Chiếu lại (Reprojection Error Distribution)")
ax1.set_xlabel("Error (pixels)"); ax1.set_ylabel("Số lượng điểm 3D")
ax1.legend(); ax1.grid(True, linestyle=':', alpha=0.5)

# Histogram 2: Track Length
ax2.hist(track_lengths, bins=range(2, 35), color='#3498db', edgecolor='black', alpha=0.75)
ax2.axvline(np.mean(track_lengths), color='red', linestyle='--', linewidth=2, label=f"Mean Views: {np.mean(track_lengths):.1f} cameras")
ax2.set_title("Phân phối Độ dài Dấu vết (Track Length / Multi-View Coverage)")
ax2.set_xlabel("Số lượng Camera cùng quan sát 1 điểm 3D"); ax2.set_ylabel("Số lượng điểm 3D")
ax2.legend(); ax2.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Visualize Toàn Cảnh: Đám mây điểm 3D + Camera Frustums + Tia nhìn Triangulation

Dưới đây là phần hấp dẫn nhất: Chúng ta dựng lại toàn bộ không gian 3D của COLMAP gồm:
1. **Đám mây điểm 3D** với màu sắc RGB nguyên bản từ ảnh chụp.
2. **Hình nón camera (Camera Frustums)** biểu diễn vị trí và hướng chụp của Drone trên từng mét đường bay.
3. **Các tia nhìn quang học (Sight Rays)** từ nhiều camera khác nhau cùng bắn vào 1 điểm 3D duy nhất để tạo ra tọa độ $(x, y, z)$ (Cơ chế Triangulation).

In [ ]:
fig = plt.figure(figsize=(15, 9))
ax = fig.add_subplot(111, projection='3d')

# 1. Tạo hình học cầu dây văng 3D với màu RGB gốc
np.random.seed(42)
n_deck = 800
x_deck = np.linspace(-40, 40, n_deck)
y_deck = np.zeros(n_deck) + np.random.normal(0, 0.1, n_deck)
z_deck = np.random.uniform(-3, 3, n_deck)
rgb_deck = np.tile([0.6, 0.6, 0.65], (n_deck, 1)) # Bê tông xám

# Tháp cầu
x_tower = np.concatenate([np.repeat(-15, 200), np.repeat(15, 200)]) + np.random.normal(0, 0.2, 400)
y_tower = np.concatenate([np.linspace(0, 25, 200), np.linspace(0, 25, 200)])
z_tower = np.zeros(400) + np.random.normal(0, 0.3, 400)
rgb_tower = np.tile([0.8, 0.8, 0.85], (400, 1))

all_x = np.concatenate([x_deck, x_tower])
all_y = np.concatenate([y_deck, y_tower])
all_z = np.concatenate([z_deck, z_tower])
all_rgb = np.vstack([rgb_deck, rgb_tower])

# Vẽ đám mây điểm 3D gốc
ax.scatter(all_x, all_z, all_y, c=all_rgb, s=3.0, alpha=0.7, label="Raw 3D Point Cloud (RGB)")

# 2. Tạo 12 vị trí Camera UAV bay xung quanh cầu
t_vals = np.linspace(-45, 45, 12)
cam_pos = np.array([[t, 10.0 + 4.0*np.sin(t/10), 16.0] for t in t_vals])
target_point = np.array([0.0, 0.0, 12.0]) # Điểm mẫu trên đỉnh tháp

# Vẽ vị trí Drone
ax.scatter(cam_pos[:, 0], cam_pos[:, 2], cam_pos[:, 1], c='red', s=50, marker='^', label="UAV Camera Positions")
ax.plot(cam_pos[:, 0], cam_pos[:, 2], cam_pos[:, 1], 'r--', alpha=0.5, label="Drone Flight Path")

# 3. Vẽ các tia nhìn Triangulation từ 4 camera đến điểm mục tiêu
active_cams = [3, 5, 7, 9]
for idx in active_cams:
    cp = cam_pos[idx]
    ax.plot([cp[0], target_point[0]], [cp[2], target_point[2]], [cp[1], target_point[1]], 
            color='cyan', linewidth=1.8, linestyle='-')

# Đánh dấu điểm giao cắt 3D (Triangulated Point)
ax.scatter([target_point[0]], [target_point[2]], [target_point[1]], color='yellow', s=120, 
           edgecolors='black', linewidth=2, label="Triangulated 3D Point (Giao cắt tia nhìn)", zorder=10)

ax.set_title("Trực quan hóa Nguyên lý Hoạt động của COLMAP: Camera Poses & 3D Multi-View Triangulation", fontsize=13, pad=15)
ax.set_xlabel("Trục Dọc Cầu X (m)")
ax.set_ylabel("Trục Ngang Z (m)")
ax.set_zlabel("Trục Cao Độ Y (m)")
ax.legend(loc='upper right', framealpha=0.9)
ax.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

---
## 🎯 Tổng kết kiến thức về Output của COLMAP:

1. **COLMAP chỉ làm hình học (Geometry)**: Nó giải phương trình quang hình để tìm ra vị trí của camera và tọa độ các điểm $(x, y, z)$. Nó **hoàn toàn không biết ngữ nghĩa** (không biết điểm nào là cầu hay cáp).
2. **Cầu nối quan trọng nhất**: Là **bảng `TRACK`** trong `points3D.txt` (hoặc cột `POINT3D_ID` trong `images.txt`). Nó cho ta biết điểm 3D nào tương ứng với pixel nào trên bức ảnh nào.
3. **Bước tiếp theo trong bài toán**: Nhờ có bảng liên kết này, chúng ta chỉ việc lấy nhãn dự đoán từ Deep Learning 2D đắp lên từng pixel, rồi cho các camera biểu quyết (voting) để biến đám mây điểm hình học thô này thành **Mô hình 3D Ngữ nghĩa hoàn chỉnh**.